# 24 - Source online threshold-refinement worker 0

This is shard 0 of 4 for the source-only causal online gate. At every decision chunk,
the source checkpoint runs the same K=5 probe at Euler steps `(3,4)` as before. At each selected
Euler step, the established **last refinement** is applied iff that step's `u_mean >= 0.03`.

Only the gated intervention is collected: 1,300 identities total (10 episodes per task), or about
325 rollouts per worker. The progress table compares its running SR with the historical **unrefined
source baseline** from `pro-16suite-k5-steps34-v1`; that column is not the old offline-thresholded
policy. Worker 0 may first use `EPISODE_LIMIT=1`; restore it to `None` for collection, and the smoke
row will be reused.

## 1. Setup a fresh GPU runtime

In [ ]:
EXTRAS = 'sim'
SETUP_ENV = True
import urllib.request
exec(urllib.request.urlopen('https://raw.githubusercontent.com/ArjunS07/cs159-sp26/main/pnp-vla/scripts/colab_bootstrap.py').read().decode())

## 2. Configuration and resumable collection

In [ ]:
from pathlib import Path
from google.colab import drive
from pnp.config import PI05_REPO_ID
from pnp.diversity import (DIVERSITY_FIXED_REFINEMENT_THRESHOLD,
    SOURCE_THRESHOLD_REFINEMENT_EXPERIMENT, load_bootstrap_manifest,
    run_source_threshold_refinement_worker)

drive.mount("/content/drive")

EPISODES_PER_TASK = 10
SHARD_COUNT = 4
SHARD_INDEX = 0
EPISODE_LIMIT = None  # worker-0 smoke: set 1 once, then restore None
THRESHOLD = DIVERSITY_FIXED_REFINEMENT_THRESHOLD  # 0.03, predeclared
EXPERIMENT = SOURCE_THRESHOLD_REFINEMENT_EXPERIMENT
MANIFEST_PATH = Path(
    "/content/drive/MyDrive/pnp_diversity_v2/bootstrap_manifest_finetuned_v2.json")
manifest = load_bootstrap_manifest(MANIFEST_PATH)
assert manifest["source_model"] == PI05_REPO_ID, manifest["source_model"]
SOURCE_MODEL_REVISION = manifest["source_model_revision"]
assert SOURCE_MODEL_REVISION, "v2 manifest is missing source_model_revision"

print({"experiment": EXPERIMENT, "threshold": THRESHOLD,
       "threshold_signal": "per-step u_mean at Euler steps 3 and 4",
       "episodes_per_task": EPISODES_PER_TASK,
       "shard_count": SHARD_COUNT, "shard_index": SHARD_INDEX,
       "episode_limit": EPISODE_LIMIT,
       "manifest_hash": manifest["manifest_hash"],
       "source_model_revision": SOURCE_MODEL_REVISION})
run_source_threshold_refinement_worker(
    episodes_per_task=EPISODES_PER_TASK, episode_limit=EPISODE_LIMIT,
    threshold=THRESHOLD, shard_count=SHARD_COUNT, shard_index=SHARD_INDEX,
    manifest_hash=manifest["manifest_hash"],
    source_model_revision=SOURCE_MODEL_REVISION, experiment=EXPERIMENT)